<a href="https://colab.research.google.com/github/Ishita-Gupta-110405/ELPV-defect-classification/blob/main/final_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Navigate to your uploaded project directory
project_dir = '/content/drive/MyDrive/dl folder'
os.chdir(project_dir)
print(f"Current working directory: {os.getcwd()}")

Mounted at /content/drive
Current working directory: /content/drive/MyDrive/dl folder


In [2]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.cuda.amp import autocast, GradScaler
from google.colab import files

# Ensure Colab can find your local src module
sys.path.append('/content/drive/MyDrive/dl folder')
from src.elpv_dataset.utils import load_dataset

class ELPV_ViT_Dataset(Dataset):
    def __init__(self, images, labels):
        # Resize to 224x224 and duplicate across 3 channels for ViT
        imgs = torch.tensor(images, dtype=torch.float32).unsqueeze(1) / 255.0
        imgs = F.interpolate(imgs, size=(224, 224), mode='bilinear', align_corners=False)
        self.images = imgs.repeat(1, 3, 1, 1)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

def train_and_export():
    print("Loading ELPV dataset from Google Drive...")
    images, proba, types = load_dataset()

    classes = np.unique(proba)
    class_mapping = {val: idx for idx, val in enumerate(classes)}
    y = np.array([class_mapping[p] for p in proba])

    X_temp, X_test, y_temp, y_test = train_test_split(images, y, test_size=0.15, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1764, stratify=y_temp, random_state=42)

    # Utilizing T4 VRAM: Increased batch size and enabled pin_memory for faster GPU transfer
    train_loader = DataLoader(ELPV_ViT_Dataset(X_train, y_train), batch_size=32, shuffle=True, pin_memory=True)
    test_loader = DataLoader(ELPV_ViT_Dataset(X_test, y_test), batch_size=32, shuffle=False, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Hardware utilized: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

    model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
    model.heads.head = nn.Linear(model.heads.head.in_features, 4)
    model = model.to(device)

    class_counts = np.bincount(y_train)
    weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    scaler = GradScaler()
    epochs = 15

    print("\nStarting Mixed Precision Training...")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()

            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")

    print("\nGenerating Final Evaluation Metrics...")
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device, non_blocking=True)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds, target_names=["0.0 (None)", "0.33 (Low)", "0.67 (Med)", "1.0 (High)"], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

    # Save the trained model to Colab's temporary environment
    save_path = '/content/vit_elpv_finetuned.pth'
    torch.save(model.state_dict(), save_path)
    print(f"\nModel saved successfully. Initiating download to local machine...")

    return save_path

# Execute training pipeline and download weights
model_path = train_and_export()
files.download(model_path)

ModuleNotFoundError: No module named 'src'

In [4]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.cuda.amp import autocast, GradScaler
from google.colab import files
from google.colab import drive

# 1. Mount Drive and Map Directory Structure
drive.mount('/content/drive')
base_dir = '/content/drive/MyDrive/dl folder'

print(f"\n--- Scanning Directory Structure in {base_dir} ---")
src_parent_path = None

for root, dirs, files_list in os.walk(base_dir):
    level = root.replace(base_dir, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    if 'src' in dirs and src_parent_path is None:
        src_parent_path = root

if not src_parent_path:
    raise FileNotFoundError("Could not find the 'src' folder. Please ensure it fully uploaded to Google Drive.")

print(f"\n-> Dynamically linking to path: {src_parent_path}")
sys.path.insert(0, src_parent_path)
os.chdir(src_parent_path)

# 2. Import Dataset Utility
from src.elpv_dataset.utils import load_dataset

# 3. Model and Training Setup
class ELPV_ViT_Dataset(Dataset):
    def __init__(self, images, labels):
        imgs = torch.tensor(images, dtype=torch.float32).unsqueeze(1) / 255.0
        imgs = F.interpolate(imgs, size=(224, 224), mode='bilinear', align_corners=False)
        self.images = imgs.repeat(1, 3, 1, 1)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

def train_and_export():
    print("\nLoading ELPV dataset into RAM...")
    images, proba, types = load_dataset()

    classes = np.unique(proba)
    class_mapping = {val: idx for idx, val in enumerate(classes)}
    y = np.array([class_mapping[p] for p in proba])

    X_temp, X_test, y_temp, y_test = train_test_split(images, y, test_size=0.15, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1764, stratify=y_temp, random_state=42)

    train_loader = DataLoader(ELPV_ViT_Dataset(X_train, y_train), batch_size=32, shuffle=True, pin_memory=True)
    test_loader = DataLoader(ELPV_ViT_Dataset(X_test, y_test), batch_size=32, shuffle=False, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Hardware utilized: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

    model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
    model.heads.head = nn.Linear(model.heads.head.in_features, 4)
    model = model.to(device)

    class_counts = np.bincount(y_train)
    weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    # Stabilized Optimizer: Lower learning rate and weight decay
    optimizer = optim.Adam(model.parameters(), lr=2e-5, weight_decay=1e-4)

    # Increased epochs for generalization and Cosine Annealing Scheduler
    epochs = 40
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()

    print(f"\nStarting Stabilized Mixed Precision Training for {epochs} Epochs...")
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()

            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()

            # Gradient Clipping to prevent loss oscillation
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f} | LR: {current_lr:.6e}")

    print("\nGenerating Final Evaluation Metrics...")
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device, non_blocking=True)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds, target_names=["0.0 (None)", "0.33 (Low)", "0.67 (Med)", "1.0 (High)"], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

    save_path = '/content/vit_elpv_finetuned.pth'
    torch.save(model.state_dict(), save_path)
    print(f"\nModel saved successfully. Initiating download to local machine...")

    return save_path

model_path = train_and_export()
files.download(model_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Scanning Directory Structure in /content/drive/MyDrive/dl folder ---
dl folder/
    dl_project/
        src/
            elpv_dataset/
                data/
                    images/
                __pycache__/
        .git/
            hooks/
            logs/
                refs/
                    remotes/
                        origin/
                    heads/
            objects/
                ff/
                fe/
                f5/
                fd/
                f6/
                f4/
                fc/
                f2/
                f8/
                fa/
                f9/
                fb/
                f7/
                f3/
                f0/
                f1/
                e5/
                ec/
                e8/
                ef/
                ee/
                eb/
                ea/
          

/tmp/ipykernel_1062/1084016930.py:82: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1062/1084016930.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Starting Stabilized Mixed Precision Training for 40 Epochs...
Epoch 1/40 - Loss: 1.2973 | LR: 1.996917e-05
Epoch 2/40 - Loss: 1.0192 | LR: 1.987688e-05
Epoch 3/40 - Loss: 0.8877 | LR: 1.972370e-05
Epoch 4/40 - Loss: 0.7587 | LR: 1.951057e-05
Epoch 5/40 - Loss: 0.6562 | LR: 1.923880e-05
Epoch 6/40 - Loss: 0.5536 | LR: 1.891007e-05
Epoch 7/40 - Loss: 0.4257 | LR: 1.852640e-05
Epoch 8/40 - Loss: 0.3253 | LR: 1.809017e-05
Epoch 9/40 - Loss: 0.2154 | LR: 1.760406e-05
Epoch 10/40 - Loss: 0.1761 | LR: 1.707107e-05
Epoch 11/40 - Loss: 0.1382 | LR: 1.649448e-05
Epoch 12/40 - Loss: 0.1191 | LR: 1.587785e-05
Epoch 13/40 - Loss: 0.0959 | LR: 1.522499e-05
Epoch 14/40 - Loss: 0.1133 | LR: 1.453990e-05
Epoch 15/40 - Loss: 0.0784 | LR: 1.382683e-05
Epoch 16/40 - Loss: 0.0828 | LR: 1.309017e-05
Epoch 17/40 - Loss: 0.0542 | LR: 1.233445e-05
Epoch 18/40 - Loss: 0.0346 | LR: 1.156434e-05
Epoch 19/40 - Loss: 0.0287 | LR: 1.078459e-05
Epoch 20/40 - Loss: 0.0343 | LR: 1.000000e-05
Epoch 21/40 - Loss: 0.0135

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.cuda.amp import autocast, GradScaler
from google.colab import files
from google.colab import drive

# 1. Mount Drive and Map Directory Structure
drive.mount('/content/drive')
base_dir = '/content/drive/MyDrive/dl folder'

print(f"\n--- Scanning Directory Structure in {base_dir} ---")
src_parent_path = None
for root, dirs, files_list in os.walk(base_dir):
    level = root.replace(base_dir, '').count(os.sep)
    if 'src' in dirs and src_parent_path is None:
        src_parent_path = root

sys.path.insert(0, src_parent_path)
os.chdir(src_parent_path)

from src.elpv_dataset.utils import load_dataset

# 2. Define Heavy Data Augmentation for Training, and Standard Prep for Val/Test
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1)) # Duplicate 1 channel to 3 for ViT
])

eval_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.repeat(3, 1, 1))
])

class ELPV_ViT_Dataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.transform = transform

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

def train_and_export():
    print("\nLoading ELPV dataset into RAM...")
    images, proba, types = load_dataset()

    classes = np.unique(proba)
    class_mapping = {val: idx for idx, val in enumerate(classes)}
    y = np.array([class_mapping[p] for p in proba])

    X_temp, X_test, y_temp, y_test = train_test_split(images, y, test_size=0.15, stratify=y, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1764, stratify=y_temp, random_state=42)

    train_loader = DataLoader(ELPV_ViT_Dataset(X_train, y_train, transform=train_transforms), batch_size=32, shuffle=True, pin_memory=True)
    val_loader = DataLoader(ELPV_ViT_Dataset(X_val, y_val, transform=eval_transforms), batch_size=32, shuffle=False, pin_memory=True)
    test_loader = DataLoader(ELPV_ViT_Dataset(X_test, y_test, transform=eval_transforms), batch_size=32, shuffle=False, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
    model.heads.head = nn.Linear(model.heads.head.in_features, 4)
    model = model.to(device)

    class_counts = np.bincount(y_train)
    weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = optim.Adam(model.parameters(), lr=2e-5, weight_decay=1e-4)
    epochs = 20
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()

    best_val_loss = float('inf')

    print("\nStarting Training with Validation Tracking...")
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()

            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        scheduler.step()

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        avg_train_loss = train_loss/len(train_loader)
        avg_val_loss = val_loss/len(val_loader)
        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.2f}%")

        # Save best weights based on validation performance to counteract overfitting
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), '/content/vit_best_val.pth')

    print("\nLoading Best Validation Weights for Final Test Evaluation...")
    model.load_state_dict(torch.load('/content/vit_best_val.pth'))

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device, non_blocking=True)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds, target_names=["0.0 (None)", "0.33 (Low)", "0.67 (Med)", "1.0 (High)"], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

    return '/content/vit_best_val.pth'

model_path = train_and_export()
files.download(model_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Scanning Directory Structure in /content/drive/MyDrive/dl folder ---

Loading ELPV dataset into RAM...


/tmp/ipykernel_1062/1464672343.py:92: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1062/1464672343.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



Starting Training with Validation Tracking...
Epoch 1/20 | Train Loss: 1.3492 Acc: 31.32% | Val Loss: 1.2866 Acc: 39.34%
Epoch 2/20 | Train Loss: 1.1421 Acc: 55.07% | Val Loss: 1.1348 Acc: 50.76%
Epoch 3/20 | Train Loss: 1.0685 Acc: 59.48% | Val Loss: 1.2158 Acc: 26.14%
Epoch 4/20 | Train Loss: 0.9740 Acc: 62.69% | Val Loss: 1.0599 Acc: 56.09%
Epoch 5/20 | Train Loss: 0.9011 Acc: 67.16% | Val Loss: 1.0713 Acc: 71.32%
Epoch 6/20 | Train Loss: 0.8769 Acc: 67.48% | Val Loss: 1.1109 Acc: 73.35%
Epoch 7/20 | Train Loss: 0.8868 Acc: 69.61% | Val Loss: 1.0943 Acc: 76.65%
Epoch 8/20 | Train Loss: 0.8302 Acc: 72.39% | Val Loss: 1.1536 Acc: 61.17%
Epoch 9/20 | Train Loss: 0.7934 Acc: 71.90% | Val Loss: 1.1538 Acc: 65.74%
Epoch 10/20 | Train Loss: 0.7584 Acc: 73.53% | Val Loss: 1.1130 Acc: 72.84%
Epoch 11/20 | Train Loss: 0.6774 Acc: 78.21% | Val Loss: 1.1656 Acc: 68.53%
Epoch 12/20 | Train Loss: 0.7064 Acc: 76.20% | Val Loss: 1.1136 Acc: 68.02%
Epoch 13/20 | Train Loss: 0.6138 Acc: 78.21% | Val

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import os
import sys
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from skimage.feature import hog

# 1. Setup Colab Path
base_dir = '/content/drive/MyDrive/dl folder'
src_parent_path = None
for root, dirs, files_list in os.walk(base_dir):
    if 'src' in dirs and src_parent_path is None:
        src_parent_path = root
if src_parent_path:
    sys.path.insert(0, src_parent_path)
    os.chdir(src_parent_path)

from preprocessing import get_splits

def extract_hog_features(img_array):
    print(f"Extracting HOG features for {len(img_array)} images...")
    features = [hog(img, pixels_per_cell=(16, 16), cells_per_block=(2, 2)) for img in img_array]
    return np.array(features)

def main_svm():
    # Fetch all splits, including the test set
    X_train, X_val, X_test, y_train, y_val, y_test = get_splits()

    X_train_hog = extract_hog_features(X_train)
    X_test_hog = extract_hog_features(X_test)

    print("Training HOG + SVM baseline (balanced class weights)...")
    svm_clf = SVC(kernel='rbf', class_weight='balanced', random_state=42)
    svm_clf.fit(X_train_hog, y_train)

    print("\n--- Final Test Set Evaluation ---")
    test_preds = svm_clf.predict(X_test_hog)

    print(classification_report(y_test, test_preds, target_names=["0.0 (None)", "0.33 (Low)", "0.67 (Med)", "1.0 (High)"], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_test, test_preds))

main_svm()

Loading ELPV dataset...
Splitting dataset...
Extracting HOG features for 1836 images...
Extracting HOG features for 394 images...
Training HOG + SVM baseline (balanced class weights)...

--- Final Test Set Evaluation ---
              precision    recall  f1-score   support

  0.0 (None)       0.80      0.71      0.75       227
  0.33 (Low)       0.46      0.52      0.49        44
  0.67 (Med)       0.14      0.31      0.20        16
  1.0 (High)       0.73      0.73      0.73       107

    accuracy                           0.68       394
   macro avg       0.53      0.57      0.54       394
weighted avg       0.71      0.68      0.69       394

Confusion Matrix:
 [[161  25  21  20]
 [ 12  23   2   7]
 [  8   1   5   2]
 [ 21   1   7  78]]


In [7]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

# 1. Setup Colab Path
base_dir = '/content/drive/MyDrive/dl folder'
src_parent_path = None
for root, dirs, files_list in os.walk(base_dir):
    if 'src' in dirs and src_parent_path is None:
        src_parent_path = root
if src_parent_path:
    sys.path.insert(0, src_parent_path)
    os.chdir(src_parent_path)

from preprocessing import get_splits

class ELPVDataset(Dataset):
    def __init__(self, images, labels):
        self.images = torch.tensor(images, dtype=torch.float32).unsqueeze(1) / 255.0
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.images[idx], self.labels[idx]

def main_resnet():
    X_train, X_val, X_test, y_train, y_val, y_test = get_splits()

    train_loader = DataLoader(ELPVDataset(X_train, y_train), batch_size=32, shuffle=True, pin_memory=True)
    val_loader = DataLoader(ELPVDataset(X_val, y_val), batch_size=32, shuffle=False, pin_memory=True)
    test_loader = DataLoader(ELPVDataset(X_test, y_test), batch_size=32, shuffle=False, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Setting up ResNet-18 on {device}...")

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    original_conv1 = model.conv1
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    with torch.no_grad():
        model.conv1.weight.copy_(original_conv1.weight.sum(dim=1, keepdim=True))
    model.fc = nn.Linear(model.fc.in_features, 4)
    model = model.to(device)

    # Weighting loss for class imbalance
    import numpy as np
    class_counts = np.bincount(y_train)
    weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    epochs = 15
    best_val_loss = float('inf')

    print("\nStarting Training (15 Epochs)...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        avg_val_loss = val_loss/len(val_loader)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), '/content/resnet_best_val.pth')

    print("\n--- Final Test Set Evaluation ---")
    model.load_state_dict(torch.load('/content/resnet_best_val.pth'))
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device, non_blocking=True)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds, target_names=["0.0 (None)", "0.33 (Low)", "0.67 (Med)", "1.0 (High)"], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

main_resnet()

Loading ELPV dataset...
Splitting dataset...
Setting up ResNet-18 on cuda...
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 175MB/s]



Starting Training (15 Epochs)...
Epoch 1/15 | Train Loss: 1.2523 | Val Loss: 1.2442
Epoch 2/15 | Train Loss: 0.8808 | Val Loss: 1.1104
Epoch 3/15 | Train Loss: 0.6473 | Val Loss: 1.2400
Epoch 4/15 | Train Loss: 0.4381 | Val Loss: 1.2625
Epoch 5/15 | Train Loss: 0.2780 | Val Loss: 1.5015
Epoch 6/15 | Train Loss: 0.1624 | Val Loss: 1.3615
Epoch 7/15 | Train Loss: 0.0899 | Val Loss: 1.5533
Epoch 8/15 | Train Loss: 0.0554 | Val Loss: 1.5201
Epoch 9/15 | Train Loss: 0.0316 | Val Loss: 1.7501
Epoch 10/15 | Train Loss: 0.0259 | Val Loss: 1.7209
Epoch 11/15 | Train Loss: 0.0202 | Val Loss: 1.7634
Epoch 12/15 | Train Loss: 0.0384 | Val Loss: 1.8928
Epoch 13/15 | Train Loss: 0.0217 | Val Loss: 1.8365
Epoch 14/15 | Train Loss: 0.0145 | Val Loss: 1.8654
Epoch 15/15 | Train Loss: 0.0088 | Val Loss: 1.8937

--- Final Test Set Evaluation ---
              precision    recall  f1-score   support

  0.0 (None)       0.87      0.40      0.55       227
  0.33 (Low)       0.27      0.59      0.37        

In [8]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# 1. Setup Colab Path
base_dir = '/content/drive/MyDrive/dl folder'
src_parent_path = None
for root, dirs, files_list in os.walk(base_dir):
    if 'src' in dirs and src_parent_path is None:
        src_parent_path = root
if src_parent_path:
    sys.path.insert(0, src_parent_path)
    os.chdir(src_parent_path)

from preprocessing import get_splits

# 2. Data Augmentation (Kept as 1-channel for ResNet)
train_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

eval_transforms = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])

class ELPVDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.transform = transform

    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        img = self.images[idx]
        if self.transform:
            img = self.transform(img)
        else:
            img = torch.tensor(img, dtype=torch.float32).unsqueeze(0) / 255.0
        return img, self.labels[idx]

def main_resnet_augmented():
    X_train, X_val, X_test, y_train, y_val, y_test = get_splits()

    train_loader = DataLoader(ELPVDataset(X_train, y_train, transform=train_transforms), batch_size=32, shuffle=True, pin_memory=True)
    val_loader = DataLoader(ELPVDataset(X_val, y_val, transform=eval_transforms), batch_size=32, shuffle=False, pin_memory=True)
    test_loader = DataLoader(ELPVDataset(X_test, y_test, transform=eval_transforms), batch_size=32, shuffle=False, pin_memory=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    original_conv1 = model.conv1
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    with torch.no_grad():
        model.conv1.weight.copy_(original_conv1.weight.sum(dim=1, keepdim=True))
    model.fc = nn.Linear(model.fc.in_features, 4)
    model = model.to(device)

    class_counts = np.bincount(y_train)
    weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4) # Added weight decay for extra regularization
    epochs = 20 # Bumped to 20 to allow learning from augmented data
    best_val_loss = float('inf')

    print("\nStarting Augmented Training (20 Epochs)...")
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

        avg_train_loss = train_loss/len(train_loader)
        avg_val_loss = val_loss/len(val_loader)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), '/content/resnet_best_val_aug.pth')

    print("\n--- Final Test Set Evaluation ---")
    model.load_state_dict(torch.load('/content/resnet_best_val_aug.pth'))
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device, non_blocking=True)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds, target_names=["0.0 (None)", "0.33 (Low)", "0.67 (Med)", "1.0 (High)"], zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))

main_resnet_augmented()

Loading ELPV dataset...
Splitting dataset...

Starting Augmented Training (20 Epochs)...
Epoch 1/20 | Train Loss: 1.2567 | Val Loss: 1.2832
Epoch 2/20 | Train Loss: 1.0984 | Val Loss: 1.0867
Epoch 3/20 | Train Loss: 1.0155 | Val Loss: 1.0759
Epoch 4/20 | Train Loss: 1.0062 | Val Loss: 1.2309
Epoch 5/20 | Train Loss: 0.9556 | Val Loss: 1.0109
Epoch 6/20 | Train Loss: 0.8998 | Val Loss: 1.0658
Epoch 7/20 | Train Loss: 0.9048 | Val Loss: 1.0932
Epoch 8/20 | Train Loss: 0.8859 | Val Loss: 1.1235
Epoch 9/20 | Train Loss: 0.8425 | Val Loss: 1.0729
Epoch 10/20 | Train Loss: 0.8347 | Val Loss: 1.0814
Epoch 11/20 | Train Loss: 0.8253 | Val Loss: 1.0300
Epoch 12/20 | Train Loss: 0.7416 | Val Loss: 1.1045
Epoch 13/20 | Train Loss: 0.7093 | Val Loss: 1.1413
Epoch 14/20 | Train Loss: 0.7730 | Val Loss: 1.2837
Epoch 15/20 | Train Loss: 0.6917 | Val Loss: 1.1366
Epoch 16/20 | Train Loss: 0.6650 | Val Loss: 1.0478
Epoch 17/20 | Train Loss: 0.6150 | Val Loss: 1.0703
Epoch 18/20 | Train Loss: 0.5905 | V